# Predicting MYH Application Approval

Trains three classification models on 8 years of MYH (Myndigheten för yrkeshögskolan)
application data (2018–2025) to estimate the probability that a new application will be
**approved** or **rejected**.

Includes hyperparameter tuning via grid search, trend analysis, and an interactive predictor.
Each section includes a plain-language explanation of the decisions made.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_PATH = Path("data/curated_applications.csv")
MODEL_OUTPUT_PATH = Path("api/model.joblib")
MODEL_COLUMNS_OUTPUT_PATH = Path("api/model_columns.json")

In [2]:
# ── Pipeline event log ────────────────────────────────────────────────────────
# Shared list populated by every step. The final section renders it as a colour-coded table.
PIPELINE_LOG: list[dict] = []


def log_event(step: str, level: str, message: str, detail: str = "") -> None:
    """Append one event to the pipeline log.

    level: "ok" | "warning" | "error"
    """
    PIPELINE_LOG.append(
        {"Step": step, "Level": level, "Message": message, "Detail": detail}
    )


# ── Setup check ───────────────────────────────────────────────────────────────
if not DATA_PATH.exists():
    log_event(
        "setup",
        "error",
        "Data file not found - run data_preparation.ipynb first",
        str(DATA_PATH),
    )
else:
    log_event("setup", "ok", "Data file found", str(DATA_PATH))

print(f"Setup: {'OK' if DATA_PATH.exists() else 'ERROR - data file missing'}")

Setup: OK


## 1. Load and Prepare Data

Three possible outcomes exist: `approved`, `rejected`, and one single `withdrawn` row.
We drop `withdrawn` (one sample cannot teach a model anything) and treat this as
**binary classification**: approved = 1, rejected = 0.

The dataset is imbalanced , 64 % rejected vs 36 % approved. All three models are
configured with `class_weight='balanced'` (or XGBoost's `scale_pos_weight`) so they
do not simply learn to predict "rejected" for everything.

In [3]:
try:
    all_applications = pd.read_csv(DATA_PATH)
    log_event("load", "ok", f"CSV read: {all_applications.shape[0]:,} rows")
except FileNotFoundError:
    log_event(
        "load",
        "error",
        "Data file not found - run data_preparation.ipynb first",
        str(DATA_PATH),
    )
    raise
except Exception as exc:
    log_event("load", "error", "Failed to read data file", str(exc))
    raise

print(f"Raw dataset: {all_applications.shape[0]:,} rows")
print(all_applications["beslut_normalized"].value_counts())
print()

binary_applications = all_applications[
    all_applications["beslut_normalized"].isin(["approved", "rejected"])
].copy()
binary_applications["target"] = (
    binary_applications["beslut_normalized"] == "approved"
).astype(int)

approved_count = binary_applications["target"].sum()
rejected_count = len(binary_applications) - approved_count
imbalance_ratio = rejected_count / approved_count

withdrawn_count = int((all_applications["beslut_normalized"] == "withdrawn").sum())
if withdrawn_count > 0:
    log_event(
        "load",
        "warning",
        f"Dropped {withdrawn_count} 'withdrawn' row(s) - too few to train on",
    )

log_event(
    "load",
    "ok",
    f"Working dataset: {len(binary_applications):,} rows",
    f"approved={approved_count:,} ({approved_count / len(binary_applications):.1%}), "
    f"rejected={rejected_count:,} ({rejected_count / len(binary_applications):.1%})",
)

print(f"Working dataset : {len(binary_applications):,} rows")
print(
    f"  Approved      : {approved_count:,} ({approved_count / len(binary_applications):.1%})"
)
print(
    f"  Rejected      : {rejected_count:,} ({rejected_count / len(binary_applications):.1%})"
)
print(f"  Imbalance ratio (rejected/approved): {imbalance_ratio:.2f}")

Raw dataset: 9,983 rows
beslut_normalized
rejected     6391
approved     3591
withdrawn       1
Name: count, dtype: int64

Working dataset : 9,982 rows
  Approved      : 3,591 (36.0%)
  Rejected      : 6,391 (64.0%)
  Imbalance ratio (rejected/approved): 1.78


## 2. Feature Engineering

**Included features:**

| Feature | Type | Why |
|---------|------|-----|
| `lan` | categorical | Region strongly influences approval rates |
| `utbildningsomrade` | categorical | Education area is MYH's primary lens |
| `studieform` | categorical | Classroom / distance / flex |
| `huvudmannatyp` | categorical | Private / municipal / regional / state operator |
| `source_year` | integer | Captures year-to-year policy changes |
| `yh_poang` | integer | Program size in YH points |
| `studietakt_procent` | integer | Study pace (50–100 %) |
| `is_distance` | boolean | Direct yes/no for distance mode |
| `has_multiple_municipalities` | boolean | Multi-municipality span |

**Excluded:**
- `utbildningsanordnare` , ~1 800 unique providers, creates thousands of dummy columns and causes overfitting
- `beslut` , raw version of the target (data leakage)
- `sun5_*`, `seqf_niva`, `smalt_yrkesomrade` , 62 % null (2023–2025 only)
- `antal_kommuner` , 1 105 nulls from 2018; `has_multiple_municipalities` covers the same signal

Categorical columns are one-hot encoded with `pd.get_dummies`. Numeric nulls are filled
with the column median; categorical nulls become `'Unknown'`.

In [4]:
CATEGORICAL_FEATURES = ["lan", "utbildningsomrade", "studieform", "huvudmannatyp"]
NUMERIC_FEATURES = ["source_year", "yh_poang", "studietakt_procent"]
BOOLEAN_FEATURES = ["is_distance", "has_multiple_municipalities"]

feature_data = binary_applications[
    CATEGORICAL_FEATURES + NUMERIC_FEATURES + BOOLEAN_FEATURES
].copy()

for column in NUMERIC_FEATURES:
    null_count = feature_data[column].isna().sum()
    if null_count > 0:
        median_value = feature_data[column].median()
        feature_data[column] = feature_data[column].fillna(median_value)
        log_event(
            "features",
            "warning",
            f"{column}: {null_count} nulls filled with median ({median_value})",
        )

for column in CATEGORICAL_FEATURES:
    null_count = feature_data[column].isna().sum()
    if null_count > 0:
        feature_data[column] = feature_data[column].fillna("Unknown")
        log_event(
            "features", "warning", f"{column}: {null_count} nulls filled with 'Unknown'"
        )

encoded_features = pd.get_dummies(
    feature_data, columns=CATEGORICAL_FEATURES, drop_first=False
)

for column in BOOLEAN_FEATURES:
    encoded_features[column] = encoded_features[column].fillna(0).astype(int)

training_columns = list(encoded_features.columns)
log_event(
    "features",
    "ok",
    f"Feature matrix: {encoded_features.shape[0]:,} rows x {encoded_features.shape[1]} columns",
)
print(
    f"Feature matrix: {encoded_features.shape[0]:,} rows x {encoded_features.shape[1]} columns"
)

Feature matrix: 9,982 rows x 49 columns


## 3. Train / Test Split

80 % of rows are used for training; 20 % are held out as an unseen test set.
`stratify=y` ensures both halves keep the same 36 / 64 class ratio.

The grid search in the next section uses 5-fold cross-validation **within** the training
set only , the 20 % test set is never seen during parameter tuning, giving an unbiased
final evaluation.

In [5]:
X = encoded_features
y = binary_applications["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

log_event(
    "split",
    "ok",
    f"Train/test split: {len(X_train):,} train, {len(X_test):,} test",
    f"Approval rate - train: {y_train.mean():.1%}, test: {y_test.mean():.1%}",
)

print(f"Training set : {len(X_train):,} rows")
print(f"Test set     : {len(X_test):,} rows")
print(f"Approval rate - train: {y_train.mean():.1%}  test: {y_test.mean():.1%}")

Training set : 7,985 rows
Test set     : 1,997 rows
Approval rate - train: 36.0%  test: 36.0%


## 4. Hyperparameter Tuning

`GridSearchCV` tries every combination of parameters in the grid and picks the one
with the highest average score across 5 cross-validation folds on the training set.

**Scoring metric:** F1 for the approved class , more informative than accuracy under
class imbalance (a model that always predicts "rejected" scores 64 % accuracy but 0 F1).

Each model is tuned independently. This section may take **2–5 minutes** to run.

In [6]:
decision_tree_param_grid = {
    "max_depth": [4, 6, 8, 10, 12],
    "min_samples_leaf": [5, 10, 20],
}

decision_tree_grid_search = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    decision_tree_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)
decision_tree_grid_search.fit(X_train, y_train)

log_event(
    "tuning",
    "ok",
    f"Decision Tree - best CV F1: {decision_tree_grid_search.best_score_:.3f}",
    str(decision_tree_grid_search.best_params_),
)

print(f"Decision Tree - best params : {decision_tree_grid_search.best_params_}")
print(f"Decision Tree - best CV F1  : {decision_tree_grid_search.best_score_:.3f}")

Decision Tree - best params : {'max_depth': 12, 'min_samples_leaf': 20}
Decision Tree - best CV F1  : 0.537


In [7]:
random_forest_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [8, 12, None],
    "max_features": ["sqrt", "log2"],
}

random_forest_grid_search = GridSearchCV(
    RandomForestClassifier(
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    random_forest_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=1,
)
random_forest_grid_search.fit(X_train, y_train)

log_event(
    "tuning",
    "ok",
    f"Random Forest - best CV F1: {random_forest_grid_search.best_score_:.3f}",
    str(random_forest_grid_search.best_params_),
)

print(f"Random Forest - best params : {random_forest_grid_search.best_params_}")
print(f"Random Forest - best CV F1  : {random_forest_grid_search.best_score_:.3f}")

Random Forest - best params : {'max_depth': 12, 'max_features': 'sqrt', 'n_estimators': 300}
Random Forest - best CV F1  : 0.559


In [8]:
xgboost_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
}

xgboost_grid_search = GridSearchCV(
    XGBClassifier(
        scale_pos_weight=imbalance_ratio,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        verbosity=0,
    ),
    xgboost_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
)
xgboost_grid_search.fit(X_train, y_train)

log_event(
    "tuning",
    "ok",
    f"XGBoost - best CV F1: {xgboost_grid_search.best_score_:.3f}",
    str(xgboost_grid_search.best_params_),
)

print(f"XGBoost - best params : {xgboost_grid_search.best_params_}")
print(f"XGBoost - best CV F1  : {xgboost_grid_search.best_score_:.3f}")

XGBoost - best params : {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 300}
XGBoost - best CV F1  : 0.572


In [9]:
grid_search_summary = pd.DataFrame(
    [
        {
            "Model": "Decision Tree",
            "Best params": str(decision_tree_grid_search.best_params_),
            "CV F1 (approved)": f"{decision_tree_grid_search.best_score_:.3f}",
        },
        {
            "Model": "Random Forest",
            "Best params": str(random_forest_grid_search.best_params_),
            "CV F1 (approved)": f"{random_forest_grid_search.best_score_:.3f}",
        },
        {
            "Model": "XGBoost",
            "Best params": str(xgboost_grid_search.best_params_),
            "CV F1 (approved)": f"{xgboost_grid_search.best_score_:.3f}",
        },
    ]
)
grid_search_summary

,Model,Best params,CV F1 (approved)
0,Decision Tree,"{'max_depth': 12, 'min_samples_leaf': 20}",0.537
1,Random Forest,"{'max_depth': 12, 'max_features': 'sqrt', 'n_e...",0.559
2,XGBoost,"{'learning_rate': 0.1, 'max_depth': 4, 'n_esti...",0.572


In [10]:
# GridSearchCV with refit=True (default) already re-trains on the full X_train
models = {
    "Decision Tree": decision_tree_grid_search.best_estimator_,
    "Random Forest": random_forest_grid_search.best_estimator_,
    "XGBoost": xgboost_grid_search.best_estimator_,
}
print("Tuned models ready.")

Tuned models ready.


## 5. Evaluate Models

**Why not just accuracy?** A model that always predicts "rejected" scores 64 % accuracy
while being completely useless. We focus on **F1 (approved)** , harmonic mean of precision
and recall for the approved class.

- **Precision (approved):** of all applications predicted approved, what fraction actually were?
- **Recall (approved):** of all applications that were actually approved, how many did we catch?
- **F1:** combines both into one score , the primary ranking metric.

In [11]:
for model_name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"{'=' * 52}")
    print(f"  {model_name}")
    print(f"{'=' * 52}")
    print(classification_report(y_test, y_pred, target_names=["rejected", "approved"]))
    print()

  Decision Tree
              precision    recall  f1-score   support

    rejected       0.74      0.63      0.68      1279
    approved       0.48      0.62      0.54       718

    accuracy                           0.62      1997
   macro avg       0.61      0.62      0.61      1997
weighted avg       0.65      0.62      0.63      1997


  Random Forest
              precision    recall  f1-score   support

    rejected       0.75      0.71      0.73      1279
    approved       0.52      0.58      0.55       718

    accuracy                           0.66      1997
   macro avg       0.64      0.64      0.64      1997
weighted avg       0.67      0.66      0.66      1997


  XGBoost
              precision    recall  f1-score   support

    rejected       0.75      0.65      0.69      1279
    approved       0.49      0.61      0.55       718

    accuracy                           0.63      1997
   macro avg       0.62      0.63      0.62      1997
weighted avg       0.66      0

In [12]:
# Top-10 feature importances per model - tabular format.
importance_rows = []
for model_name, model in models.items():
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[::-1][:10]
    for rank, idx in enumerate(top_indices, 1):
        importance_rows.append(
            {
                "Model": model_name,
                "Rank": rank,
                "Feature": training_columns[idx],
                "Importance": round(float(importances[idx]), 4),
            }
        )

importance_df = pd.DataFrame(importance_rows)
for model_name in models:
    model_slice = importance_df[importance_df["Model"] == model_name].set_index("Rank")
    display(
        model_slice[["Feature", "Importance"]]
        .style.background_gradient(subset=["Importance"], cmap="Blues")
        .format({"Importance": "{:.4f}"})
        .set_caption(f"Top 10 Feature Importances - {model_name}")
        .set_table_styles(
            [
                {
                    "selector": "caption",
                    "props": [
                        ("font-size", "0.95em"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                        ("padding-bottom", "8px"),
                    ],
                }
            ]
        )
    )

,Feature,Importance
Rank,,
1,utbildningsomrade_Teknik och tillverkning,0.1723
2,source_year,0.1466
3,yh_poang,0.1402
4,huvudmannatyp_Privat,0.0966
5,utbildningsomrade_Data/IT,0.0759
6,"utbildningsomrade_Ekonomi, administration och försäljning",0.0726
7,studieform_Bunden,0.0696
8,lan_Skåne,0.0281
9,lan_Stockholm,0.0274


,Feature,Importance
Rank,,
1,yh_poang,0.1632
2,source_year,0.1555
3,utbildningsomrade_Teknik och tillverkning,0.0845
4,utbildningsomrade_Data/IT,0.0639
5,huvudmannatyp_Privat,0.0522
6,"utbildningsomrade_Ekonomi, administration och försäljning",0.0456
7,huvudmannatyp_Kommun,0.0373
8,studietakt_procent,0.0276
9,studieform_Bunden,0.0272


,Feature,Importance
Rank,,
1,utbildningsomrade_Teknik och tillverkning,0.1118
2,huvudmannatyp_Privat,0.1004
3,utbildningsomrade_Data/IT,0.0662
4,"utbildningsomrade_Ekonomi, administration och försäljning",0.0459
5,is_distance,0.0397
6,lan_Jönköping,0.0352
7,utbildningsomrade_Säkerhetstjänster,0.0333
8,lan_Flera kommuner,0.0292
9,utbildningsomrade_Hälso- och sjukvård samt socialt arbete,0.0262


In [13]:
comparison_rows = []
for model_name, model in models.items():
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    f1_approved = report["1"]["f1-score"]
    test_accuracy = accuracy_score(y_test, y_pred)
    comparison_rows.append(
        {
            "Model": model_name,
            "Accuracy": f"{test_accuracy:.3f}",
            "F1 (approved)": f"{f1_approved:.3f}",
            "Recall (approved)": f"{report['1']['recall']:.3f}",
            "F1 (rejected)": f"{report['0']['f1-score']:.3f}",
            "Recall (rejected)": f"{report['0']['recall']:.3f}",
        }
    )
    log_event(
        "evaluate",
        "ok",
        f"{model_name}: accuracy={test_accuracy:.3f}, F1(approved)={f1_approved:.3f}",
    )

pd.DataFrame(comparison_rows)

,Model,Accuracy,F1 (approved),Recall (approved),F1 (rejected),Recall (rejected)
0,Decision Tree,0.623,0.540,0.616,0.681,0.628
1,Random Forest,0.659,0.548,0.575,0.726,0.706
2,XGBoost,0.634,0.546,0.611,0.694,0.647


## 6. Approval Trends (2018–2025)

These charts use raw historical counts , not model predictions.
They show how competitive MYH has become over time and which education areas
have the best and worst track records.

In [14]:
yearly_stats = (
    binary_applications.groupby("source_year")["target"]
    .agg(total_applications="count", approved_applications="sum")
    .assign(
        approval_rate_pct=lambda df: (
            df["approved_applications"] / df["total_applications"] * 100
        ).round(1)
    )
    .reset_index()
)
print(yearly_stats.to_string(index=False))

 source_year  total_applications  approved_applications  approval_rate_pct
        2018                1105                    496               44.9
        2019                1237                    482               39.0
        2020                1482                    484               32.7
        2021                1238                    426               34.4
        2022                1207                    420               34.8
        2023                1258                    477               37.9
        2024                1272                    344               27.0
        2025                1183                    462               39.1


In [15]:
area_rates = (
    binary_applications.groupby("utbildningsomrade")["target"]
    .agg(total_applications="count", approved_applications="sum")
    .assign(
        approval_rate_pct=lambda df: (
            df["approved_applications"] / df["total_applications"] * 100
        ).round(1)
    )
    .sort_values("approval_rate_pct", ascending=False)
)

overall_mean = area_rates["approval_rate_pct"].mean()

display(
    area_rates.style.background_gradient(
        subset=["approval_rate_pct"], cmap="RdYlGn", vmin=15, vmax=55
    )
    .format(
        {
            "total_applications": "{:,}",
            "approved_applications": "{:,}",
            "approval_rate_pct": "{:.1f}%",
        }
    )
    .set_caption(
        f"Approval Rate by Education Area (2018-2025)  |  Average: {overall_mean:.1f}%"
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("font-size", "0.95em"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("padding-bottom", "8px"),
                ],
            }
        ]
    )
)

,total_applications,approved_applications,approval_rate_pct
utbildningsomrade,,,
Miljövård och miljöskydd,1,1,100.0%
Övrigt,10,7,70.0%
Teknik och tillverkning,"1,510",835,55.3%
"Lantbruk, djurvård, trädgård, skog och fiske",255,130,51.0%
Transporttjänster,216,107,49.5%
"Kultur, media och design",377,168,44.6%
Hälso- och sjukvård samt socialt arbete,"1,172",475,40.5%
Samhällsbyggnad och byggteknik,"1,343",540,40.2%
"Hotell, restaurang och turism",313,121,38.7%


## 7. Save Best Model

The model with the highest F1 on the approved class is saved to `api/model.joblib`.
The exact column list is saved to `api/model_columns.json` so a future API endpoint
can reconstruct the same feature matrix from new user inputs.

In [16]:
best_model_name = max(
    models,
    key=lambda name: classification_report(
        y_test, models[name].predict(X_test), output_dict=True
    )["1"]["f1-score"],
)
best_model = models[best_model_name]
print(f"Best model by F1 (approved): {best_model_name}")

MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

try:
    joblib.dump(best_model, MODEL_OUTPUT_PATH)
    log_event("save", "ok", f"Model saved: {best_model_name}", str(MODEL_OUTPUT_PATH))
    print(f"Model saved   -> {MODEL_OUTPUT_PATH}")
except Exception as exc:
    log_event("save", "error", "Failed to save model", str(exc))
    raise

try:
    # Explicit UTF-8 encoding avoids cp1252 issues on Windows (Swedish chars in column names).
    MODEL_COLUMNS_OUTPUT_PATH.write_text(
        json.dumps(training_columns, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    log_event(
        "save",
        "ok",
        f"Column list saved: {len(training_columns)} features",
        str(MODEL_COLUMNS_OUTPUT_PATH),
    )
    print(f"Columns saved -> {MODEL_COLUMNS_OUTPUT_PATH}")
except Exception as exc:
    log_event("save", "error", "Failed to save column list", str(exc))
    raise

Best model by F1 (approved): Random Forest
Model saved   -> api\model.joblib
Columns saved -> api\model_columns.json


## 8. Interactive Predictor

Run the **valid values** cell first to see all accepted inputs, then edit
`my_application` and run the predictor cell.

> Values must match **exactly** (case-sensitive) the strings printed below.
> An unknown value gets a zero in its dummy column , the model falls back to
> other features, which reduces confidence.

In [17]:
print("Valid values for each categorical field:\n")
for column in CATEGORICAL_FEATURES:
    unique_values = sorted(binary_applications[column].dropna().unique())
    print(f"{column}:")
    for value in unique_values:
        print(f"  '{value}'")
    print()

Valid values for each categorical field:

lan:
  'Blekinge'
  'Dalarna'
  'Flera kommuner'
  'Gotland'
  'Gävleborg'
  'Halland'
  'Jämtland'
  'Jönköping'
  'Kalmar'
  'Kronoberg'
  'Norrbotten'
  'Skåne'
  'Stockholm'
  'Södermanland'
  'Uppsala'
  'Värmland'
  'Västerbotten'
  'Västernorrland'
  'Västmanland'
  'Västra Götaland'
  'Örebro'
  'Östergötland'

utbildningsomrade:
  'Data/IT'
  'Ekonomi, administration och försäljning'
  'Friskvård och kroppsvård'
  'Hotell, restaurang och turism'
  'Hälso- och sjukvård samt socialt arbete'
  'Journalistik och information'
  'Juridik'
  'Kultur, media och design'
  'Lantbruk, djurvård, trädgård, skog och fiske'
  'Miljövård och miljöskydd'
  'Pedagogik och undervisning'
  'Samhällsbyggnad och byggteknik'
  'Säkerhetstjänster'
  'Teknik och tillverkning'
  'Transporttjänster'
  'Övrigt'

studieform:
  'Bunden'
  'Distans'

huvudmannatyp:
  'Kommun'
  'Privat'
  'Region'
  'Statlig'



In [ ]:
my_application = {
    "lan": "Stockholm",
    "utbildningsomrade": "Teknik och tillverkning",
    "studieform": "Distans",
    "huvudmannatyp": "Privat",
    "source_year": 2025,
    "yh_poang": 300,
    "studietakt_procent": 100,
    "is_distance": 1,
    "has_multiple_municipalities": 0,
}

input_row = pd.DataFrame([my_application])
input_encoded = pd.get_dummies(
    input_row, columns=CATEGORICAL_FEATURES, drop_first=False
)
input_aligned = input_encoded.reindex(columns=training_columns, fill_value=0)

approval_probability = best_model.predict_proba(input_aligned)[0][1]
rejection_probability = 1 - approval_probability

# Pre-build the progress bar outside the f-string so the literal block
# characters (U+2588 full, U+2591 light) do not need to be expressed as
# backslash escapes inside an f-string expression (which is a 3.12+ feature).
bar_total = 40
bar_filled = int(approval_probability * bar_total)
bar = "█" * bar_filled + "░" * (bar_total - bar_filled)

print(f"Model: {best_model_name}")
print()
print(f"  Approval probability : {approval_probability:.1%}")
print(f"  Rejection probability: {rejection_probability:.1%}")
print()
print(f"  [{bar}] {approval_probability:.1%}")

## Limitations

1. **Not MYH's actual process.** MYH evaluates program quality, regional need, and employer
   demand , none of which are in this dataset. The model approximates statistical patterns,
   not the real decision criteria.

2. **Policy changes break predictions.** If MYH shifts priorities (e.g. a push toward
   green tech), the model will not know until it is retrained. `source_year` captures
   gradual drift but not sudden policy turns.

3. **Sparse categories have low confidence.** Areas or counties with few historical
   applications produce wide uncertainty in the probability estimate.

4. **Provider quality is invisible.** A school with a strong track record may genuinely
   have better odds , the model cannot see this because provider name was excluded.

5. **Probabilities are estimates, not guarantees.** A 60 % approval score means similar
   past applications were often approved , not that this specific one will be.

---
## 9. Pipeline Report

A full record of every event logged during this run - setup, data load, feature engineering, model training, evaluation, and save.
Green = ok, amber = warning, red = error.

This is the single place to look when something goes wrong. A warning here is informative; an error here means the saved model or output should not be trusted.

In [19]:
if not PIPELINE_LOG:
    print("No events logged.")
else:
    log_df = pd.DataFrame(PIPELINE_LOG)

    ok_count = int((log_df["Level"] == "ok").sum())
    warn_count = int((log_df["Level"] == "warning").sum())
    err_count = int((log_df["Level"] == "error").sum())

    print(
        f"Pipeline complete  |  {ok_count} ok  |  {warn_count} warning(s)  |  {err_count} error(s)"
    )
    if err_count:
        print("  -> Action required: review the errors before using the saved model.")
    elif warn_count:
        print("  -> Warnings present: model may be usable but check the details.")
    else:
        print("  -> All events clean - model is ready for use.")
    print()

    def _style_level(value: str) -> str:
        if value == "ok":
            return "background-color: rgba(46, 204, 113, 0.18)"
        if value == "warning":
            return "background-color: rgba(230, 126, 34, 0.28); font-weight: 600"
        if value == "error":
            return "background-color: rgba(231, 76, 60, 0.28); font-weight: 600"
        return ""

    display(log_df.style.map(_style_level, subset=["Level"]))

Pipeline complete  |  13 ok  |  1 warning(s)  |  0 error(s)
  -> Warnings present: model may be usable but check the details.



,Step,Level,Message,Detail
0,setup,ok,Data file found,data\curated_applications.csv
1,load,ok,"CSV read: 9,983 rows",
2,load,warning,Dropped 1 'withdrawn' row(s) - too few to train on,
3,load,ok,"Working dataset: 9,982 rows","approved=3,591 (36.0%), rejected=6,391 (64.0%)"
4,features,ok,"Feature matrix: 9,982 rows x 49 columns",
5,split,ok,"Train/test split: 7,985 train, 1,997 test","Approval rate - train: 36.0%, test: 36.0%"
6,tuning,ok,Decision Tree - best CV F1: 0.537,"{'max_depth': 12, 'min_samples_leaf': 20}"
7,tuning,ok,Random Forest - best CV F1: 0.559,"{'max_depth': 12, 'max_features': 'sqrt', 'n_estimators': 300}"
8,tuning,ok,XGBoost - best CV F1: 0.572,"{'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 300}"
9,evaluate,ok,"Decision Tree: accuracy=0.623, F1(approved)=0.540",
